In [1]:
pip install xlrd


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

df = pd.read_excel("GSAF5.xls")
df.shape

(7125, 23)

In [3]:
# Ver nulos por columna
df.isna().sum()

Date                 0
Year                 2
Type                18
Country             50
State              487
Location           567
Activity           583
Name               218
Sex                578
Age               2994
Injury              36
Fatal Y/N          561
Time              3527
Species           3131
Source              20
pdf                326
href formula       331
href               329
Case Number        327
Case Number.1      328
original order     326
Unnamed: 21       7124
Unnamed: 22       7123
dtype: int64

In [4]:
# Ver duplicados por filas
df.duplicated().sum()

np.int64(0)

In [5]:
# Eliminar filas y columnas vacías
df_clean = df.dropna(axis=1, how="all")   
df_clean = df_clean.dropna(axis=0, how="all")   
df_clean.columns = df_clean.columns.str.strip()
df_clean.shape

(7125, 23)

In [6]:
# Elimino duplicados
df_clean = df_clean.drop_duplicates()
df_clean.shape

(7125, 23)

In [7]:
# Elimino los nulos que me molestan de las columnas de mi hipotesis
df_clean = df_clean.dropna(subset=["Country"])
df_clean.shape

(7075, 23)

In [8]:
# Elimino los nulos de la columna Species y los reemplazo por Unknown, ya que son más de la mitad de las filas y no quiero perder información de otras columnas.
df_clean["Species"] = df_clean["Species"].fillna("Unknown")

In [9]:
# Elimino las columnas que no aportan (Unnamed, notas internas y las de códigos)
columnas_fuera = ["Unnamed: 21", "Unnamed: 22", "pdf", "href formula", "href",
                  "Case Number", "Case Number.1", "original order", "Source", "Name"]
df_clean = df_clean.drop(columns=columnas_fuera)
df_clean.shape

(7075, 13)

In [10]:
# Me quedo solo con  años fiables, desde antes de 1900 los hay mucho sin valor
df_clean["Year"] = pd.to_numeric(df_clean["Year"], errors="coerce")
df_clean = df_clean[df_clean["Year"] >= 1900]
df_clean.shape

(6349, 13)

In [11]:
# Limpio "Country"
print("Países distintos antes:", df_clean["Country"].nunique())

df_clean["Country"] = (df_clean["Country"]
                       .str.strip()
                       .str.upper()
                       .str.replace("?", "", regex=False)
                       .str.strip())

print("Países distintos después:", df_clean["Country"].nunique())
df_clean["Country"].value_counts().head(10)


Países distintos antes: 223
Países distintos después: 181


Country
USA                 2447
AUSTRALIA           1377
SOUTH AFRICA         569
BAHAMAS              137
PAPUA NEW GUINEA     132
BRAZIL               121
NEW ZEALAND          115
MEXICO                95
NEW CALEDONIA         66
ITALY                 62
Name: count, dtype: int64

In [12]:
# Limpio "Species"
import re
def limpiar_especie(texto):
    """Agrupa el texto libre de Species en una especie conocida."""
    texto = str(texto).lower()
    if texto == "unknown":
        return "Unknown"
    if re.search(r"not confirmed|unconfirmed|invalid|questionable", texto):
        return "Not confirmed"
    patrones = {
        "Sand tiger shark": r"sand tiger|raggedtooth|grey nurse",
        "White shark": r"white",
        "Tiger shark": r"tiger",
        "Bull shark": r"bull",
        "Blacktip shark": r"blacktip|black tip",
        "Nurse shark": r"nurse",
        "Bronze whaler": r"bronze|whaler",
        "Mako shark": r"mako",
        "Hammerhead": r"hammerhead",
    }
    for especie, patron in patrones.items():
        if re.search(patron, texto):
            return especie
    return "Unknown"

df_clean["Species_clean"] = df_clean["Species"].apply(limpiar_especie)
df_clean["Species_clean"].value_counts()

Species_clean
Unknown             4122
White shark          750
Not confirmed        421
Tiger shark          338
Bull shark           237
Blacktip shark       134
Sand tiger shark      88
Bronze whaler         86
Nurse shark           71
Mako shark            57
Hammerhead            45
Name: count, dtype: int64

In [13]:
# Termino de mirar "State"
df_clean["State"].value_counts().head(15)


State
Florida                  1168
New South Wales           459
Hawaii                    325
Queensland                323
California                322
Western Australia         234
KwaZulu-Natal             204
Western Cape Province     189
Eastern Cape Province     161
South Carolina            158
South Australia           111
North Carolina            110
Pernambuco                 80
Texas                      79
Victoria                   79
Name: count, dtype: int64

In [15]:
# Busco países escritos de distintas formas
print("Lista ordenada:")
print(sorted(df_clean["Country"].unique()))

print("\nVariantes de palabras sospechosas:")
print(df_clean[df_clean["Country"].str.contains("MARTIN|TRINIDAD|TOBAGO", na=False)]["Country"].value_counts())

print("\nPaíses que aparecen muy poco:")
print(df_clean["Country"].value_counts().tail(40))

Lista ordenada:
['ADMIRALTY ISLANDS', 'AMERICAN SAMOA', 'ANDAMAN / NICOBAR ISLANDAS', 'ANGOLA', 'ANTIGUA', 'ARGENTINA', 'ARUBA', 'ATLANTIC OCEAN', 'AUSTRALIA', 'AZORES', 'BAHAMAS', 'BANGLADESH', 'BARBADOS', 'BAY OF BENGAL', 'BELIZE', 'BERMUDA', 'BRAZIL', 'BRITISH ISLES', 'BRITISH OVERSEAS TERRITORY', 'BRITISH VIRGIN ISLANDS', 'BRITISH WEST INDIES', 'CANADA', 'CANARY ISLANDS', 'CAPE VERDE', 'CARIBBEAN SEA', 'CAYMAN ISLANDS', 'CENTRAL PACIFIC', 'CHILE', 'CHINA', 'COLOMBIA', 'COLUMBIA', 'COMOROS', 'CORAL SEA', 'COSTA RICA', 'CRETE', 'CROATIA', 'CUBA', 'CURACAO', 'CYPRUS', 'DIEGO GARCIA', 'DOMINICAN REPUBLIC', 'ECUADOR', 'EGYPT', 'EGYPT / ISRAEL', 'EL SALVADOR', 'ENGLAND', 'FALKLAND ISLANDS', 'FEDERATED STATES OF MICRONESIA', 'FIJI', 'FRANCE', 'FRENCH POLYNESIA', 'GABON', 'GALAPOGOS ISLANDS', 'GRAND CAYMAN', 'GREECE', 'GRENADA', 'GUAM', 'GUATEMALA', 'GULF OF ADEN', 'GUYANA', 'HAITI', 'HAWAII', 'HONDURAS', 'HONG KONG', 'ICELAND', 'INDIA', 'INDIAN OCEAN', 'INDONESIA', 'IRAN', 'IRAN / IRAQ', 

In [16]:
# Unifico países escritos de distintas formas
correcciones_pais = {
    "COLUMBIA": "COLOMBIA",
    "REUNION ISLAND": "REUNION",
    "ST. MARTIN": "ST MARTIN",
    "ST. MAARTIN": "ST MARTIN",
    "TURKS AND CAICOS": "TURKS & CAICOS",
    "UNITED ARAB EMIRATES (UAE)": "UNITED ARAB EMIRATES",
    "MALDIVE ISLANDS": "MALDIVES",
    "TRINIDAD": "TRINIDAD & TOBAGO",
    "TOBAGO": "TRINIDAD & TOBAGO",
    "ENGLAND": "UNITED KINGDOM",
    "SCOTLAND": "UNITED KINGDOM",
    "HAWAII": "USA",
    "GRAND CAYMAN": "CAYMAN ISLANDS",
    "WESTERN SAMOA": "SAMOA",
}
df_clean["Country"] = df_clean["Country"].replace(correcciones_pais)
print("Países distintos:", df_clean["Country"].nunique())

Países distintos: 167


In [17]:
# Extraigo el primer número que aparezca en Age ("30s" -> 30, "mid-20s" -> 20, "Ca. 33" -> 33)
df_clean["Age_clean"] = df_clean["Age"].astype(str).str.extract(r"(\d+)")[0]
df_clean["Age_clean"] = pd.to_numeric(df_clean["Age_clean"], errors="coerce")

# Las edades imposibles (menos de 1 o más de 100) las dejo como nulo
df_clean.loc[(df_clean["Age_clean"] < 1) | (df_clean["Age_clean"] > 100), "Age_clean"] = None

df_clean["Age_clean"].describe()

count    3956.000000
mean       28.381951
std        14.745938
min         1.000000
25%        17.000000
50%        25.000000
75%        37.000000
max        87.000000
Name: Age_clean, dtype: float64

In [19]:
# Creo grupos de edad para el análisis (útil para el plan familiar)
rangos = [0, 12, 17, 30, 50, 100]
etiquetas = ["Niño (0-12)", "Adolescente (13-17)", "Joven (18-30)", "Adulto (31-50)", "Mayor (51+)"]

df_clean["Age_group"] = pd.cut(df_clean["Age_clean"], bins=rangos, labels=etiquetas)
df_clean["Age_group"].value_counts()

Age_group
Joven (18-30)          1515
Adulto (31-50)         1018
Adolescente (13-17)     690
Mayor (51+)             388
Niño (0-12)             345
Name: count, dtype: int64